# 🌤️ PROJE 5 — Hava: Malatya + 3 İl + İklim Değişikliği
## Mini Proje — Araştırma Projeniz
*Bugün (Gün 2) seçtiğiniz proje — kampın son günü "Bilimsel Yöntem" oturumunda sunulacak*

**Veri seti:** Malatya, İstanbul, Erzurum 2020-2024 (toplam 5481 günlük gözlem)
**Kaynak:** Open-Meteo (ERA5 yeniden analizi) — GERÇEK verilerdir

### 🎯 Sizin Göreviniz
Hava modellerini eğitmek + **iklim değişikliği gözlemi** + il karşılaştırma!

**Aşağıdaki seçeneklerden en az 3'ünü deneyin:**
1. **1-gün vs 3-gün tahmin** karşılaştırması (RMSE artışı)
2. **Mevsim bazında** model performansı
3. 🌡️ **İklim değişikliği:** 2020 → 2024 arası sıcaklık değişti mi? (3 il)
4. 🔥 **Aşırı sıcaklık olayları:** >35°C kaç gün, yıllar arası fark
5. 🌧️ **Yağış-sıcaklık ilişkisi:** Yağmurlu günler gerçekten serin mi?
6. 🏙️ **3 il karşılaştırma:** Malatya vs İstanbul vs Erzurum
7. **Tek model 3 il için çalışır mı?** Eğitim Malatya, test İstanbul/Erzurum (transfer öğrenimi)

---

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/STEM_yaz_kampi_2026')
from python_code.helper_tr import *
import pandas as pd
import numpy as np

hava = pd.read_csv('input_data/iller_hava.csv')
hava['tarih'] = pd.to_datetime(hava['tarih'])
print(f'\nİl sayısı: {hava["il"].nunique()}')
print(f'Toplam: {len(hava)} günlük gözlem')
print(f'Yıl aralığı: {hava["yil"].min()} - {hava["yil"].max()}')

---
### 👥 Grup Tartışması — Başlamadan Önce
- Malatya, İstanbul ve Erzurum'dan hangisi en **sıcak**, hangisi en **soğuk**, hangisi en **değişken** sizce? Grupça tahmin edin, sonra veriye bakacağız.
- Yaşadığınız yerin havası son yıllarda değişti mi? Büyükleriniz ne diyor?

## Adım 1: 3 İl Genel Karşılaştırma 🏙️

In [ ]:
kutu_grafigi(hava, kategori_sutunu='il', sayisal_sutun='sicaklik_ort',
             baslik='İl Bazlı Sıcaklık Dağılımı (5 yıl)')

In [ ]:
ozet = hava.groupby('il').agg({
    'sicaklik_ort': ['mean', 'min', 'max', 'std'],
    'yagis': 'sum',
}).round(1)
print(ozet)

**Görev:** Hangi il en sıcak? En soğuk? En değişken (std en yüksek)? En yağışlı?

## 🌡️ Adım 2: İklim Değişikliği Gözlemi (2020 → 2024)

**Hipotez:** 5 yıl içinde sıcaklık trendi yukarı mı, aşağı mı, sabit mi?

In [ ]:
yillik = hava.groupby(['il', 'yil'])['sicaklik_ort'].mean().reset_index()
import plotly.express as px
fig = px.line(yillik, x='yil', y='sicaklik_ort', color='il', markers=True,
              title='3 İl — Yıllık Ortalama Sıcaklık (2020-2024)')
fig.show()

In [ ]:
for il in hava['il'].unique():
    veri_2020 = hava[(hava['il']==il) & (hava['yil']==2020)]['sicaklik_ort'].mean()
    veri_2024 = hava[(hava['il']==il) & (hava['yil']==2024)]['sicaklik_ort'].mean()
    print(f'{il:10}: 2020={veri_2020:.2f}°C, 2024={veri_2024:.2f}°C, Δ={veri_2024-veri_2020:+.2f}°C')

**Görev:** Hangi il en çok ısındı? Bu "sadece şans mı" yoksa "trend mi"? 5 yıl trend için yeterli mi? (Gerçek iklim biliminde 30 yıl gerek)

---
### 👥 Grup Tartışması
- Sadece **5 yıllık** veriyle 'iklim değişiyor' demek doğru mu? Bir trendi güvenle söylemek için ne kadar veri gerekir?
- 'Bu yıl çok sıcaktı' demek ile 'iklim ısınıyor' demek aynı şey mi? **Hava** (weather) ile **iklim** (climate) farkını tartışın.

## 🔥 Adım 3: Aşırı Sıcaklık Olayları

In [ ]:
asiri = hava[hava['sicaklik_max'] > 35].groupby(['il', 'yil']).size().reset_index(name='asiri_gun')
fig = px.bar(asiri, x='yil', y='asiri_gun', color='il', barmode='group',
             title='Yıllık >35°C Gün Sayısı')
fig.show()
print('\nİl × Yıl tablosu:')
print(asiri.pivot(index='yil', columns='il', values='asiri_gun').fillna(0).astype(int))

**Görev:** Hangi il en çok aşırı sıcak gün yaşıyor? 2020→2024 arasında trend?

## 🌧️ Adım 4: Yağış-Sıcaklık İlişkisi

In [ ]:
for il in hava['il'].unique():
    df_il = hava[hava['il'] == il]
    yagmurlu = df_il[df_il['yagis'] > 1]['sicaklik_ort'].mean()
    kuru = df_il[df_il['yagis'] == 0]['sicaklik_ort'].mean()
    print(f'{il:10}: Yağmurlu günler {yagmurlu:.1f}°C, kuru günler {kuru:.1f}°C, fark {kuru-yagmurlu:+.1f}°C')

## Adım 5: 1-Gün ve 3-Gün Tahmin Modeli (Malatya)

In [ ]:
def model_hazirla(df_il):
    df_il = df_il.sort_values('tarih').reset_index(drop=True)
    for g in [1, 2, 3, 7]:
        df_il[f'sicaklik_oncesi_{g}gun'] = df_il['sicaklik_ort'].shift(g)
    df_il['hedef_3gun'] = df_il['sicaklik_ort'].shift(-3)
    return df_il.dropna()

malatya = model_hazirla(hava[hava['il'] == 'Malatya'].copy())
ozellikler = ['sicaklik_oncesi_1gun', 'sicaklik_oncesi_2gun',
              'sicaklik_oncesi_3gun', 'sicaklik_oncesi_7gun', 'ay', 'yagis']
X = malatya[ozellikler]
maske = malatya['yil'] < 2024

from sklearn.metrics import mean_squared_error
m_1 = lineer_regresyon_egit(X[maske], malatya['sicaklik_ort'][maske])
rmse_1 = np.sqrt(mean_squared_error(malatya['sicaklik_ort'][~maske], m_1.predict(X[~maske])))
print(f'\n1-gün RMSE: {rmse_1:.2f}°C')

m_3 = lineer_regresyon_egit(X[maske], malatya['hedef_3gun'][maske])
rmse_3 = np.sqrt(mean_squared_error(malatya['hedef_3gun'][~maske], m_3.predict(X[~maske])))
print(f'3-gün RMSE: {rmse_3:.2f}°C')
print(f'\n3-gün tahmin {rmse_3/rmse_1:.1f}x daha zor!')

## Adım 6: Mevsim Bazında Performans

In [ ]:
test = malatya[~maske].copy()
test['tahmin'] = m_1.predict(test[ozellikler])
for mev in ['Kış', 'İlkbahar', 'Yaz', 'Sonbahar']:
    alt = test[test['mevsim'] == mev]
    rmse = np.sqrt(mean_squared_error(alt['sicaklik_ort'], alt['tahmin']))
    print(f'{mev:10}: RMSE = {rmse:.2f}°C ({len(alt)} gün)')

## 🏙️ Adım 7: 3-İl Modeli — Transfer Öğrenimi

**Hipotez:** Malatya'da eğitilmiş model İstanbul'u tahmin edebilir mi? Yoksa her il için ayrı model şart mı?

In [ ]:
tum_iller = []
for il in hava['il'].unique():
    df_il = model_hazirla(hava[hava['il'] == il].copy())
    tum_iller.append(df_il)
tum = pd.concat(tum_iller, ignore_index=True)

ml = tum[tum['il'] == 'Malatya']
ml_eg = ml[ml['yil'] < 2024]
ml_te = ml[ml['yil'] == 2024]
m_a = lineer_regresyon_egit(ml_eg[ozellikler], ml_eg['sicaklik_ort'])
rmse_a = np.sqrt(mean_squared_error(ml_te['sicaklik_ort'], m_a.predict(ml_te[ozellikler])))
print(f'A) Malatya → Malatya RMSE: {rmse_a:.2f}°C')

ist_te = tum[(tum['il']=='İstanbul') & (tum['yil']==2024)]
rmse_b = np.sqrt(mean_squared_error(ist_te['sicaklik_ort'], m_a.predict(ist_te[ozellikler])))
print(f'B) Malatya → İstanbul RMSE: {rmse_b:.2f}°C')

erz_te = tum[(tum['il']=='Erzurum') & (tum['yil']==2024)]
rmse_c = np.sqrt(mean_squared_error(erz_te['sicaklik_ort'], m_a.predict(erz_te[ozellikler])))
print(f'C) Malatya → Erzurum RMSE: {rmse_c:.2f}°C')

print(f'\nTransfer maliyeti: İstanbul {rmse_b/rmse_a:.1f}x, Erzurum {rmse_c/rmse_a:.1f}x kötü')

**Görev:** Bu "transfer öğrenimi" sonucu ne anlatır? Niye Malatya modeli Erzurum'da daha çok şaşırır?

---
### 👥 Grup Tartışması
- Malatya'da eğitilen model Erzurum'da da çalıştı. Bu sizi **şaşırttı** mı? Bir modelin başka bir şehirde çalışması ne zaman **işe yarar**, ne zaman **tehlikeli** olur?
- Model hangi şehirde en çok yanılır sizce? Neden?

## Adım 8: Sıcak Hava Dalgası — En Sıcak 5 Gün

In [ ]:
test_2024 = malatya[~maske].copy()
test_2024['tahmin'] = m_1.predict(test_2024[ozellikler])
en_sicak = test_2024.sort_values('sicaklik_ort', ascending=False).head(5)
print('2024 Malatya en sıcak 5 gün:')
print(en_sicak[['tarih', 'sicaklik_ort', 'tahmin']].assign(
    hata=lambda d: (d['sicaklik_ort'] - d['tahmin']).round(1)
).to_string(index=False))

---
### 👥 Grup Tartışması — Sunuma Hazırlık
- En güçlü bulgunuz hangisi — 3 il karşılaştırması, ısınma trendi, yoksa tahmin modeli mi? Grupça **bir ana mesaj** seçin.
- Sununuzu 'Peki bu neden önemli?' sorusuna cevap verecek şekilde bitirin.

## Adım 9: Sunum için notlarınız

- 3 il sıcaklık ortalaması: Malatya ___°C, İstanbul ___°C, Erzurum ___°C
- 🌡️ 2020 → 2024 değişim: Malatya +___°C, İstanbul +___°C, Erzurum +___°C
- 🔥 Yıllık aşırı sıcak gün (>35°C) — en yüksek il: ___
- 🌧️ Malatya yağmurlu günler kuru günlerden ___ °C daha serin
- 1-gün RMSE: ___°C, 3-gün RMSE: ___°C
- En zor mevsim: ___
- 🏙️ Malatya modeli Erzurum'da **___ kat** daha kötü performans
- **İklim değişikliği yorumu:** _________________________
- **Pratik uygulama:** Malatya kayısı bahçesine geç don tahmini için bu model nasıl kullanılır?

📝 Bilimsel yöntem şablonunu doldurun!

---
## 🚀 Hızlı Bitirenler İçin BONUS

1. **Yağış tahmini:** Sıcaklık yerine yağışı tahmin etmeye çalışın (sınıflandırma: yağacak/yağmayacak)
2. **Mevsim geçiş günleri:** En çok hatanın olduğu günler hangi tarihlerde toplanıyor? (İlkbaharda cephe geçişleri tipik)
3. **2024 yaz dalgası:** Temmuz-Ağustos 2024'te ardışık 5+ gün >35°C var mı? (Eurostat ısı dalgası tanımı)
4. **Erzurum-İstanbul sıcaklık farkı:** Yıl içinde nasıl değişiyor — kışın mı, yazın mı en fazla?
5. **Çiftçi danışmanı:** Malatya kayısı için don ihtimali yüksek hangi ay-gün kombinasyonu?

## 🆘 Yardım — Sık Karşılaşılan Hatalar

**❌ `il` sütunu yok hatası** → `iller_hava.csv` kullanıyor musunuz? `malatya_hava.csv`'de il sütunu yok

**❌ Transfer öğrenimi yüksek RMSE** → Beklenen! Erzurum dağ, Malatya step iklim. Sıcaklık aralıkları farklı

**❌ Yıllık trend net değil** → 5 yıl çok kısa. Gerçek iklim trendleri 30+ yılda anlaşılır. "5 yıl yetmez mi?" sorusu kendisi pedagojik

**❌ Tahmin grafiği boş** → `dropna()` öncesi/sonrası boyut kontrolü

**❌ R² negatif** → Test verisi yanlış seçilmiş (örn. tüm test yaza, eğitim kışa)

---
# 🧑‍🏫 Öğretmen Rehberi & Cevap Anahtarı
*Bu bölüm öğretmen içindir: beklenen sonuçlar, sık takılma noktaları ve tartışma soruları. Öğrenciler projeyi yaptıktan sonra bakabilir.*

# 🎓 PROJE 5 EĞİTMEN — Hava (Malatya Gelişmiş Tahmin)
## Mini Proje Saha Rehberi

**Bu notebook eğitmenler içindir.**

## 👥 Bu Projeyi Kim Seçer?
- Dünkü hava modelini beğenmiş, geliştirmek isteyen gruplar
- Pratik/günlük problem sevenler
- Yerel (Malatya) odaklı çalışmaktan zevk alanlar

## 🎯 Beklenen Çıktı (2 saat sonunda)
- 1-gün ve 3-gün tahmini RMSE karşılaştırılmış
- Mevsim bazında performans analizi
- **3 il (Malatya, İstanbul, Erzurum) verisiyle "transfer" deneyi:** Malatya'da eğitilen model başka illerde ne kadar çalışıyor?
- Bir "en sıcak günler" örneği analiz edilmiş

## 📊 Beklenen Sayısal Sonuçlar (`input_data/iller_hava.csv`)
| Tahmin | RMSE |
|--------|------|
| 1-gün sonrası | ~1.5 °C |
| 3-gün sonrası | ~3.5 °C (≈2.4 kat zor) |
| Mevsim bazında | ~1.4–1.7 °C (oldukça dengeli) |
| Transfer: Malatya → İstanbul | ~1.5 °C (neredeyse aynı!) |
| Transfer: Malatya → Erzurum | ~1.6 °C (biraz daha zor) |

**Önemli mesaj 1:** "3-gün, 1-günden ~2.4 kat zor" — bu bilim insanı için temel bir gözlem.

**Önemli mesaj 2 (asıl sürpriz):** Sadece "son günlerin sıcaklığı" ile çalışan model, hiç görmediği İstanbul/Erzurum'da da neredeyse aynı başarıyla çalışır — çünkü "hava yavaş değişir" kuralı her şehirde geçerli. Konuma değil, **fiziğe** dayanan bir model evrenseldir.

## ⚠️ Tipik Takılma Noktaları

**1. NaN'lar her yere dağılmış**
→ `shift()` her sütun için en başta NaN üretir. `.dropna()` çağırınca veri çok küçülürse hesabı sorgulayın

**2. Zaman serisi rastgele bölme yapıyor**
→ `egitim_test_bol(...)` rastgele bölüyor — zaman serisinde YANLIŞ! Geçmişten geleceği tahmin için tarih bazlı bölme şart. Öğrencilere "yıl < 2024" maskesini hatırlatın

**3. "Modelim çok iyi (RMSE 0.5) ama değil mi?"**
→ Genelde data leakage: hedef sütununu yanlışlıkla öznitelik olarak kullandılar. `hedef_3gun` veya `sicaklik_ort` X'te olmasın

**4. Mevsim sütunu zaten var, ama kullanmamışlar**
→ Hatırlatın: `hava['mevsim']` zaten dolu. `pd.get_dummies` ile öznitelik olarak ekleyebilirler

## 🎯 Derinleştirme Soruları

1. "En sıcak 5 gün ile en soğuk 5 günü ayırın. Model hangisini daha iyi tahmin ediyor?"
2. "Yağmurlu günlerle kuru günleri ayırın. Model birinde daha iyi mi?"
3. "Sadece son 1 yıl veriyle eğitirseniz performans nasıl?" (Az veri sorusu)
4. "Yarın hava 30°C olacak diye tahmin ettin ama gerçekte 35°C oldu. Modelin önceki 7 gün öncesinde yanılgıya işaret eden bir kalıp var mı?"
5. "Malatya'da eğitilen model İstanbul ve Erzurum'da neden hâlâ iyi çalışıyor? Hangi ilde en çok bozulur, neden?" (transfer / genelleme sezgisi)

## 💭 Etik / Sosyal Tartışma

> "Bu modelin tahmininin %95 doğru olduğunu söylesek, çiftçi yarın don olacak diye bütün domateslerini örtmeli mi?"

Yönlendirme:
- Maliyet asimetrisi (don yetişmesin = ürün kaybı $$$, gereksiz örtme = işçilik)
- 2°C hata kritik kararlarda yetersiz
- Ne kadar veri/sensör eklersek hassasiyet ne kadar artar?

## 🎤 Sunum Koçluğu

Sunum kalpleri:
- 1-gün vs 3-gün **net karşılaştırma**
- Mevsim haritası (4 RMSE değeri)
- 1 "hata büyüktü" örneği (cephe geçişi günü gibi)
- Pratik soru: "Ben yarın spor yapayım mı?" — model yarınki sıcaklığa kaç emin?